
# ToolCall-200M — Colab Micro Shard Pipeline

## What this notebook does

This notebook runs the **next stage after tokenizer training**:

```text
frozen SentencePiece tokenizer
    -> streamed raw datasets
    -> native SentencePiece tokenization
    -> packed uint16 token shard
    -> shard inspection and decoded sample checks
```

## Where to run this

Run this notebook in **Google Colab CPU runtime**.

Use **Colab High-RAM** if available, but a GPU is **not required** for this step.

| Step | Environment |
|---|---|
| Upload/extract pipeline files | Colab CPU |
| Install dependencies | Colab CPU |
| Build 1M-token micro shard | Colab CPU / RAM / network-bound |
| Inspect decoded samples | Colab CPU |
| Tiny model training later | Colab T4 GPU |
| 200M model training later | Kaggle 2×T4 GPU |

## Stop condition

After the 1M-token shard is created and inspected, **stop and review the decoded samples**. Do not build 50M tokens until the micro shard looks correct.


In [ ]:

# Runtime check
# Recommended: Colab CPU runtime. GPU is not needed for this notebook.

import os
import sys
import subprocess
from pathlib import Path

print("Python:", sys.version)
print("Current directory:", os.getcwd())

try:
    gpu_info = subprocess.check_output(
        "nvidia-smi --query-gpu=name --format=csv,noheader",
        shell=True,
        stderr=subprocess.STDOUT,
        text=True,
    ).strip()
    print("GPU detected:", gpu_info)
    print("Note: GPU is not needed for this shard-preparation notebook.")
except Exception:
    print("No GPU detected. This is fine for this notebook.")


## 1. Choose storage

Use Google Drive if you want the generated shard to persist after the Colab session ends. For a 1M-token micro shard, local `/content` is also fine.

In [ ]:

# Choose whether to persist files in Google Drive.
# For the 1M micro shard, False is fine.
# For 50M+ shards, True is recommended.

USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/toolcall_200m_next_pipeline')
else:
    PROJECT_DIR = Path('/content/toolcall_200m_next_pipeline')

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print('PROJECT_DIR:', PROJECT_DIR)


## 2. Upload and extract `toolcall_next_pipeline.zip`

Upload the zip file you downloaded from ChatGPT: `toolcall_next_pipeline.zip`.

This contains:

```text
scripts/prepare_pretraining_shards.py
scripts/inspect_token_shards.py
scripts/train_tiny_smoke.py
requirements.txt
configs/
```

In [ ]:

from google.colab import files
import zipfile
import shutil

print('Upload toolcall_next_pipeline.zip')
uploaded = files.upload()

zip_candidates = [name for name in uploaded.keys() if name.endswith('.zip')]
if not zip_candidates:
    raise FileNotFoundError('No .zip file uploaded. Please upload toolcall_next_pipeline.zip')

pipeline_zip = Path(zip_candidates[0])
print('Uploaded:', pipeline_zip)

extract_root = Path('/content/pipeline_extract')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(pipeline_zip, 'r') as zf:
    zf.extractall(extract_root)

# Detect whether zip has a top-level folder or files directly.
required = ['requirements.txt', 'scripts']
possible_roots = [extract_root] + [p for p in extract_root.iterdir() if p.is_dir()]
source_root = None
for candidate in possible_roots:
    if all((candidate / item).exists() for item in required):
        source_root = candidate
        break

if source_root is None:
    raise RuntimeError('Could not find requirements.txt and scripts/ inside uploaded zip.')

# Copy scaffold into PROJECT_DIR.
for item in source_root.iterdir():
    dest = PROJECT_DIR / item.name
    if dest.exists():
        if dest.is_dir():
            shutil.rmtree(dest)
        else:
            dest.unlink()
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print('Pipeline extracted to:', PROJECT_DIR)
print('Contents:')
for p in sorted(PROJECT_DIR.iterdir()):
    print(' -', p.name)


## 3. Install dependencies

This installs only CPU/data-preparation dependencies. It may take a few minutes.

In [ ]:

import os
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

!pip -q install -r requirements.txt
print('Dependencies installed.')


## 4. Upload tokenizer artifacts

Upload either:

1. `toolcall_spm_32k.model` and `toolcall_spm_32k.vocab`, **or**
2. the tokenizer artifact zip you created earlier.

The notebook will place the files into:

```text
tokenizer/toolcall_spm_32k.model
tokenizer/toolcall_spm_32k.vocab
```

In [ ]:

from google.colab import files
import zipfile
import shutil

TOKENIZER_DIR = PROJECT_DIR / 'tokenizer'
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

print('Upload tokenizer .model/.vocab files OR tokenizer artifact .zip')
uploaded = files.upload()

upload_names = list(uploaded.keys())

# If a zip is uploaded, extract it and search for tokenizer files.
zip_files = [Path(name) for name in upload_names if name.endswith('.zip')]

if zip_files:
    tok_extract = Path('/content/tokenizer_extract')
    if tok_extract.exists():
        shutil.rmtree(tok_extract)
    tok_extract.mkdir(parents=True, exist_ok=True)

    for zip_path in zip_files:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(tok_extract)

    model_matches = list(tok_extract.rglob('toolcall_spm_32k.model'))
    vocab_matches = list(tok_extract.rglob('toolcall_spm_32k.vocab'))

    if not model_matches:
        raise FileNotFoundError('Could not find toolcall_spm_32k.model inside uploaded zip.')

    shutil.copy2(model_matches[0], TOKENIZER_DIR / 'toolcall_spm_32k.model')

    if vocab_matches:
        shutil.copy2(vocab_matches[0], TOKENIZER_DIR / 'toolcall_spm_32k.vocab')

# Also copy directly uploaded .model/.vocab files if present.
for name in upload_names:
    path = Path(name)
    if path.name == 'toolcall_spm_32k.model':
        shutil.copy2(path, TOKENIZER_DIR / 'toolcall_spm_32k.model')
    elif path.name == 'toolcall_spm_32k.vocab':
        shutil.copy2(path, TOKENIZER_DIR / 'toolcall_spm_32k.vocab')

MODEL_PATH = TOKENIZER_DIR / 'toolcall_spm_32k.model'
VOCAB_PATH = TOKENIZER_DIR / 'toolcall_spm_32k.vocab'

if not MODEL_PATH.exists():
    raise FileNotFoundError('Missing tokenizer model: tokenizer/toolcall_spm_32k.model')

print('Tokenizer directory:', TOKENIZER_DIR)
print('Model exists:', MODEL_PATH.exists(), MODEL_PATH)
print('Vocab exists:', VOCAB_PATH.exists(), VOCAB_PATH)


## 5. Verify tokenizer before shard creation

This checks that the native SentencePiece tokenizer loads and returns non-zero token counts.

In [ ]:

import sentencepiece as spm

sp = spm.SentencePieceProcessor(model_file=str(MODEL_PATH))
print('Vocab size:', sp.vocab_size())
print('EOS ID:', sp.eos_id())

TEST_STRINGS = [
    'calendar.create_event',
    '2026-06-25T15:30:00+05:30',
    '{"decision":"call","tool_name":"calendar.create_event","arguments":{"title":"Project sync"}}',
    '<|user|>
Find unread emails from HR.
<|tool_schema|>
{"name":"email.search_emails"}
<|tool_call|>',
]

for text in TEST_STRINGS:
    ids = sp.encode(text, out_type=int)
    pieces = sp.encode(text, out_type=str)
    print('
' + '=' * 80)
    print(text)
    print('tokens:', len(ids))
    print('pieces:', pieces[:80])
    if len(ids) == 0:
        raise RuntimeError(f'Tokenizer returned zero tokens for: {text}')

if sp.vocab_size() > 65535:
    raise RuntimeError('Vocab does not fit in uint16.')

print('
Tokenizer check passed.')


## 6. Build the 1M-token micro shard

### What we are doing

Creating a tiny packed token dataset to validate the full preprocessing path.

### Where this runs

**Colab CPU/RAM.** GPU is not needed.

### Expected time

Usually a few minutes, depending on dataset streaming speed and OpenAPI repository cloning.

### Important

`--include-xlam false` avoids gated HF dataset issues for the first run.

In [ ]:

import subprocess
import shlex

MICRO_DIR = PROJECT_DIR / 'data' / 'tokenized' / 'micro_1m'

cmd = [
    'python', 'scripts/prepare_pretraining_shards.py',
    '--tokenizer', str(MODEL_PATH),
    '--output-dir', str(MICRO_DIR),
    '--target-tokens', '1000000',
    '--shard-tokens', '1000000',
    '--include-xlam', 'false',
    '--decode-samples', '5',
]

print('Running command:')
print(' '.join(shlex.quote(x) for x in cmd))

result = subprocess.run(cmd, cwd=str(PROJECT_DIR), text=True)
if result.returncode != 0:
    raise RuntimeError(f'Micro shard creation failed with exit code {result.returncode}')

print('
Micro shard creation complete:', MICRO_DIR)


## 7. Inspect the micro shard

### What we are checking

- shard exists
- token IDs are below vocab size
- decoded samples are readable
- no corrupt or empty shard files

In [ ]:

cmd = [
    'python', 'scripts/inspect_token_shards.py',
    '--data-dir', str(MICRO_DIR),
    '--tokenizer', str(MODEL_PATH),
    '--num-samples', '5',
    '--sample-tokens', '256',
]

print('Running command:')
print(' '.join(shlex.quote(x) for x in cmd))

result = subprocess.run(cmd, cwd=str(PROJECT_DIR), text=True)
if result.returncode != 0:
    raise RuntimeError(f'Shard inspection failed with exit code {result.returncode}')


## 8. Read manifest summary

This gives a compact view of what was produced.

In [ ]:

import json
import pandas as pd

manifest_path = MICRO_DIR / 'manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError(f'Missing manifest: {manifest_path}')

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))

print('Actual tokens:', manifest['actual_tokens'])
print('Target tokens:', manifest['target_tokens'])
print('Vocab size:', manifest['vocab_size'])
print('Dtype:', manifest['dtype'])
print('Number of shards:', len(manifest['shards']))
print('Tokens/sec:', round(manifest['tokens_per_second'], 2))

source_df = pd.DataFrame(manifest['source_stats'])
display(source_df)

shard_df = pd.DataFrame(manifest['shards'])
display(shard_df)


## 9. Zip and download the micro shard output

This is optional, but useful if you are not using Google Drive.

In [ ]:

import shutil
from google.colab import files

zip_base = PROJECT_DIR / 'micro_1m_token_shards'
zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format='zip',
    root_dir=str(MICRO_DIR),
)

print('Created:', zip_path)
print('Size MB:', Path(zip_path).stat().st_size / 1024**2)

DOWNLOAD_MICRO_ZIP = True
if DOWNLOAD_MICRO_ZIP:
    files.download(zip_path)


# Stop here

Do **not** run 50M shard creation until the following are true:

1. `inspect_token_shards.py` completed without errors.
2. `max_id < vocab_size`.
3. Decoded samples look like readable text / JSON / code / API docs.
4. `manifest.json` shows approximately 1,000,000 actual tokens.

Once this passes, the next step is a **50M-token smoke shard**, still in Colab CPU/High-RAM.

## Optional next step: 50M smoke shard

Leave this disabled until the 1M micro shard has passed inspection.

### Where to run

Colab CPU/High-RAM. GPU is still not required.

In [ ]:

RUN_50M_SMOKE = False

if RUN_50M_SMOKE:
    SMOKE_DIR = PROJECT_DIR / 'data' / 'tokenized' / 'smoke_50m'

    cmd = [
        'python', 'scripts/prepare_pretraining_shards.py',
        '--tokenizer', str(MODEL_PATH),
        '--output-dir', str(SMOKE_DIR),
        '--target-tokens', '50000000',
        '--shard-tokens', '10000000',
        '--include-xlam', 'false',
        '--decode-samples', '5',
    ]

    print('Running command:')
    print(' '.join(shlex.quote(x) for x in cmd))

    result = subprocess.run(cmd, cwd=str(PROJECT_DIR), text=True)
    if result.returncode != 0:
        raise RuntimeError(f'50M smoke shard creation failed with exit code {result.returncode}')

    print('50M smoke shard complete:', SMOKE_DIR)
else:
    print('50M smoke shard is disabled. Set RUN_50M_SMOKE = True only after micro shard inspection passes.')
